# 03 — Prepare & Export (v2)

> **AI-Assisted Development** — This project was built with [Kiro](https://kiro.dev). See `SOURCES.md` for full attribution.

Build the master states analysis table (one row per state, all metrics),
and package the sellable dataset.

**Pipeline:**
1. Join all 10+ source tables via SQL into one master row-per-state
2. Compute per-capita and per-VMT rates
3. Include BAC testing rates, mandatory testing laws, and enforcement procedures
4. Export to CSV + Excel + Parquet + codebook

**Output:** `export/dui_by_state_v4` — 51 rows × 48 columns

### Changes from v1 → v2
- **Added:** BAC testing rates (killed/surviving/all drivers, test type breakdown)
- **Added:** Mandatory testing law classification (29 mandatory, 22 probable-cause)
- **Added:** NASID enforcement procedures (checkpoints, no-refusal, PBT, IID, lookback, felony, high-BAC, etc.)
- **Added:** Cleaner per-VMT rate columns using NHTSA imputed fatalities
- **Removed:** Stale 2015–2020 trend export (truncated data no longer meaningful)
- **Removed:** v1 export files (superseded)

### Approach
The v3 export is built by `scripts/prepare_export_v3.py` using a single large
SQL JOIN in DuckDB. This notebook documents and inspects that output.

---
**To rebuild the export:**
```bash
/opt/anaconda3/envs/data_projects/bin/python scripts/prepare_export_v3.py
```

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

DB = Path("../data/project.duckdb")
con = duckdb.connect(str(DB), read_only=True)
print(f"Tables: {len(con.execute('SHOW TABLES').df())}")
print(f"DuckDB: {DB.resolve()}")

---
## 1. Tables joined into v2 master

The export SQL joins these tables on `state_fips` or `state_name`:

| Table | Columns contributed | Join key |
|-------|--------------------|---------|
| `states` | FIPS, name, region, pop, lat/lng, FARS raw fatalities | Primary (state_fips) |
| `nhtsa_imputed_2024` | Imputed alcohol fatalities, pct, high-BAC | state_name |
| `fars_bac_testing_2024` | Testing rates (killed/surviving/all), test type | state_fips |
| `bac_testing_laws` | Mandatory law, scope, authority | state_fips |
| `nasid_enforcement_clean` | 11 enforcement boolean/categoricals | state_fips |
| `dui_enforcement` | Criminal refusal penalty, BAC limit | state_fips |
| `dui_criminal_status_clean` | First offense felony flag | state_fips |
| `vehicle_impound_laws` | Impound, forfeiture, plate impound by state | state_fips |

In [ ]:
# Load the exported v4 dataset
master = pd.read_parquet("../export/dui_by_state_v4.parquet")
print(f"v2 export: {master.shape[0]} rows × {master.shape[1]} columns")
print(f"\nColumn groups:")
print(f"  Identity/geo: state_fips → lng (8 cols)")
print(f"  Fatality outcomes: traffic_fatalities_2024 → pct_high_bac_2024 (8 cols)")
print(f"  Rates: total_fatality_rate_per_100k, alcohol_fatality_rate_per_100k (2 cols)")
print(f"  BAC testing: pct_bac_known_killed → pct_blood_test (4 cols)")
print(f"  Testing laws: mandatory_testing_law → testing_authority (3 cols)")
print(f"  Enforcement: checkpoints_permitted → testing_methods (13 cols)")
print(f"  Other enforcement: criminal_refusal_penalty → first_offense_felony (3 cols)")
print(f"  Vehicle sanctions: vehicle_impound_law → has_mandatory_impound (7 cols)")

In [ ]:
# Full column listing
print(f"All {len(master.columns)} columns:")
for i, c in enumerate(master.columns, 1):
    dtype = master[c].dtype
    nulls = master[c].isnull().sum()
    print(f"  {i:2d}. {c:45s} {str(dtype):10s} nulls={nulls}")

---
## 2. Key metrics — descriptive stats

In [ ]:
# Fatality rates
rate_cols = ['total_fatality_rate_per_100k', 'alcohol_fatality_rate_per_100k',
             ]
master[rate_cols].describe().round(2)

In [ ]:
# BAC testing rates
test_cols = ['pct_bac_known_killed', 'pct_bac_known_all_drivers', 'pct_bac_known_surviving']
master[test_cols].describe().round(1)

---
## 3. Enforcement comparisons

Quick cross-tabs showing how enforcement features relate to outcomes.

In [ ]:
# Mandatory testing law vs fatality rates
print("=== Mandatory BAC Testing Law vs Alcohol Fatality Rate ===")
print(master.groupby('mandatory_testing_law')[['alcohol_fatality_rate_per_100k',
    'alcohol_fatality_rate_per_100k', 'pct_bac_known_killed']].mean().round(2))
print()

# Checkpoints vs fatality rates
print("=== Sobriety Checkpoints vs Alcohol Fatality Rate ===")
print(master.groupby('checkpoints_permitted')[['alcohol_fatality_rate_per_100k',
    'alcohol_fatality_rate_per_100k']].mean().round(2))
print()

# IID all-offender vs fatality rates
print("=== IID All-Offender Mandate vs Alcohol Fatality Rate ===")
print(master.groupby('iid_all_offender')[['alcohol_fatality_rate_per_100k',
    'alcohol_fatality_rate_per_100k']].mean().round(2))
print()

# No-refusal programs
print("=== No-Refusal Program Status vs Alcohol Fatality Rate ===")
print(master.groupby('no_refusal_status')[['alcohol_fatality_rate_per_100k',
    'alcohol_fatality_rate_per_100k']].mean().round(2))

In [ ]:
# Top 10 states by alcohol fatality rate per 100M VMT
print("=== Top 10 — Highest alcohol fatality rate per 100M VMT ===")
master.nlargest(10, 'alcohol_fatality_rate_per_100k')[
    ['state_abbr', 'alcohol_fatality_rate_per_100k', 'pct_alcohol_nhtsa_imputed',
     'mandatory_testing_law', 'checkpoints_permitted', 'iid_all_offender', 'no_refusal_active']
]

In [ ]:
# Bottom 10 (safest)
print("=== Top 10 — Lowest alcohol fatality rate per 100M VMT ===")
master.nsmallest(10, 'alcohol_fatality_rate_per_100k')[
    ['state_abbr', 'alcohol_fatality_rate_per_100k', 'pct_alcohol_nhtsa_imputed',
     'mandatory_testing_law', 'checkpoints_permitted', 'iid_all_offender', 'no_refusal_active']
]

---
## 4. Export products

The v3 export was built by `scripts/prepare_export_v3.py` and lives in `export/`:

| File | Format | Size |
|------|--------|------|
| `dui_by_state_v4.csv` | CSV | 51 rows × 48 cols |
| `dui_by_state_v4.xlsx` | Excel | Single sheet |
| `dui_by_state_v4.parquet` | Apache Parquet | Typed columns |
| `dui_by_state_v4_codebook.md` | Markdown | Full column descriptions + caveats |

### What was removed from export/
- `dui_by_state_v1.*` — superseded by v2 (49 cols → 57 cols)
- `dui_trends_2015_2020.*` — stale truncated data (2015-2020 only, methodology break at 2021 made it misleading)

### Full 2015–2024 trends
The complete FARS trend data (all 10 years, 510 rows) remains in DuckDB as
`fars_trends_clean`. It's not exported because the 2021 methodology change
makes cross-year comparison misleading without proper annotation. Use the
`impairment_method` column to filter if you need time series.

In [ ]:
# Verify export files exist and match
from pathlib import Path

export_dir = Path("../export")
for f in sorted(export_dir.glob("dui_by_state_v4*")):
    print(f"  {f.name:40s} {f.stat().st_size / 1024:.1f} KB")

# Verify CSV matches parquet
csv_df = pd.read_csv(export_dir / "dui_by_state_v4.csv")
assert csv_df.shape == master.shape, f"Shape mismatch: CSV {csv_df.shape} vs parquet {master.shape}"
print(f"\n✓ All export formats consistent: {master.shape[0]} rows × {master.shape[1]} cols")

---
## 5. South Carolina investigation

SC appears as the #1 or #2 outlier in NHTSA imputed alcohol fatality rate.
Investigation showed this is **NOT** due to different reporting/testing:

- SC's killed-driver BAC testing rate is **76.4%** (above national avg of 66.8%)
- SC has a **mandatory testing law** (coroner statute §17-7-80)
- The outlier status comes from SC's crash characteristics: high rates of
  nighttime single-vehicle crashes, low seatbelt use, rural high-speed roads
- The NHTSA imputation model uses these crash characteristics to assign
  alcohol probability to untested cases, pushing SC's imputed % to 40%
  vs the 24% raw FARS coding

In [ ]:
sc = master[master['state_abbr'] == 'SC'].T
sc.columns = ['South Carolina']
sc

---
## Next steps

The v2 dataset is ready for analysis. Suggested directions:

1. **Enforcement score model:** Combine checkpoints + no-refusal + IID + PBT into a
   composite "enforcement intensity" score, test against fatality rates
2. **Expected vs actual:** Predict fatality rate from VMT/pop/region, show which states
   are better/worse than expected (controls for structural factors)
3. **Measurement artifact analysis:** Does BAC testing rate predict imputed %?
   (If yes, low-testing states may be artificially inflated by the model)
4. **Regional patterns:** Aggregate enforcement profiles by Census division

See `05-analysis.ipynb` for deeper statistical work.

In [ ]:
con.close()